# 03 — Exploratory Data Analysis

**Project:** Ontology-Guided Hypothesis Generation Using LLMs and Topic Modeling in mHealth Research

Comprehensive EDA of the relevance-filtered mHealth dataset.
All visualizations are saved to `reports/figures/` for use in project reports.

**Date:** 2026-08-29

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import os

# Optional imports
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print('WordCloud not available, skipping word cloud visualizations.')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RELEVANT_FILE = os.path.join(PROJECT_ROOT, 'data', 'processed', 'mhealth_relevant_dataset.csv')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete.')

WordCloud not available, skipping word cloud visualizations.
Setup complete.


In [2]:
df = pd.read_csv(RELEVANT_FILE)
print(f'Total relevant papers: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Total relevant papers: 1750
Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms', 'journal', 'authors', 'doi', 'document', 'cleaned_text', 'source', 'keyword_score', 'matched_keywords', 'mesh_score', 'matched_mesh', 'tfidf_score', 'relevance_score', 'relevance_category', 'relevance_reason']


,pmid,title,abstract,year,mesh_terms,journal,authors,doi,document,cleaned_text,source,keyword_score,matched_keywords,mesh_score,matched_mesh,tfidf_score,relevance_score,relevance_category,relevance_reason
0,39776481,Benchmarking the clinical outcomes of Healthen...,BACKGROUND: Software as a Medical Device (SaMD...,2024.0,Humans; Chronic Disease/therapy; *Telemedicine...,Frontiers in public health,Kyriazakos S; Pnevmatikakis A; Kostopoulou K; ...,10.3389/fpubh.2024.1488687,Benchmarking the clinical outcomes of Healthen...,benchmarking the clinical outcomes of healthen...,expanded,0.40,mhealth; mobile health; health app; remote pat...,1.0,telemedicine; mobile applications; mobile app,0.728352,0.678505,Relevant,Moderate keyword match (0.40); Good MeSH align...
1,30419185,Going digital: a narrative overview of the eff...,Objective Smartphone health applications (apps...,2020.0,Chronic Disease/*therapy; Humans; *Mobile Appl...,Australian health review : a publication of th...,Scott IA; Scuffham P; Gupta D; Harch TM; Borch...,10.1071/AH18064,Going digital: a narrative overview of the eff...,going digital a narrative overview of the effe...,expanded,0.24,health app; health application; smartphone health,0.7,mobile applications; mobile app,0.405406,0.427622,Possibly relevant,Moderate keyword match (0.24); Good MeSH align...
2,36841348,Remote patient monitoring for management of di...,BACKGROUND: Diabetes mellitus is a common medi...,2023.0,"Pregnancy; Infant; Female; Humans; Infant, New...",American journal of obstetrics and gynecology,Kantorowska A; Cohen K; Oberlander M; Jaysing ...,10.1016/j.ajog.2023.02.015,Remote patient monitoring for management of di...,remote patient monitoring for management of di...,expanded,0.54,mobile health; remote patient monitoring; elec...,0.1,NaN,0.392043,0.363613,Possibly relevant,Strong keyword match (0.54); MeSH score: 0.10;...


## 1. Papers by Publication Year

In [3]:
df['year'] = df['year'].astype(str)
year_counts = df['year'].value_counts().sort_index()
valid_years = {y: c for y, c in year_counts.items() if y.isdigit() and len(y) == 4}
valid_years = dict(sorted(valid_years.items()))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = sns.color_palette('viridis', len(valid_years))
bars = axes[0].bar(valid_years.keys(), valid_years.values(), color=colors, edgecolor='white')
axes[0].set_xlabel('Publication Year')
axes[0].set_ylabel('Number of Papers')
axes[0].set_title('mHealth Papers by Publication Year', fontsize=14, fontweight='bold')
for bar, val in zip(bars, valid_years.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 3,
                 str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Line chart (cumulative growth)
cumulative = np.cumsum(list(valid_years.values()))
axes[1].plot(list(valid_years.keys()), cumulative, 'o-', color='#2ecc71', linewidth=2.5, markersize=8)
axes[1].fill_between(list(valid_years.keys()), cumulative, alpha=0.2, color='#2ecc71')
axes[1].set_xlabel('Publication Year')
axes[1].set_ylabel('Cumulative Papers')
axes[1].set_title('Growth of mHealth Research Over Time', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'papers_by_year.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_13456\4213430088.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Most Frequent MeSH Terms

In [4]:
# Extract and count MeSH terms
all_mesh = []
for terms in df['mesh_terms'].dropna():
    if str(terms).strip():
        for term in str(terms).split(';'):
            # Remove asterisks (major topic indicators) for counting
            cleaned = term.strip().lstrip('*').strip()
            if cleaned and cleaned.lower() != 'humans':
                all_mesh.append(cleaned)

mesh_counts = Counter(all_mesh)
top_mesh = pd.DataFrame(mesh_counts.most_common(20), columns=['MeSH Term', 'Count'])

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('coolwarm', len(top_mesh))
ax.barh(top_mesh['MeSH Term'][::-1], top_mesh['Count'][::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Frequency')
ax.set_title('Top 20 MeSH Terms in mHealth Literature\n(excluding "Humans")', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'top_mesh_terms.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_13456\429089971.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Most Frequent Terms in Titles

In [5]:
# Extract meaningful terms from titles
stopwords = {'a', 'an', 'the', 'and', 'or', 'of', 'in', 'to', 'for', 'on', 'with',
             'is', 'are', 'was', 'were', 'be', 'been', 'by', 'at', 'from', 'as',
             'that', 'this', 'it', 'its', 'not', 'but', 'if', 'than', 'so', 'no',
             'do', 'did', 'has', 'have', 'had', 'will', 'would', 'can', 'could',
             'should', 'may', 'might', 'shall', 'about', 'up', 'out', 'into',
             'through', 'during', 'before', 'after', 'between', 'under', 'over',
             'each', 'all', 'both', 'other', 'such', 'how', 'what', 'which',
             'who', 'when', 'where', 'why', 'also', 'more', 'most', 'very',
             'using', 'based', 'among', 'study', 'review', 'analysis', 'use',
             'their', 'we', 'our', 'us', 'them', 'they', 'he', 'she'}

all_words = []
for title in df['title'].dropna():
    words = re.findall(r'[a-z]+', str(title).lower())
    for w in words:
        if len(w) > 2 and w not in stopwords:
            all_words.append(w)

word_counts = Counter(all_words)
top_words = pd.DataFrame(word_counts.most_common(20), columns=['Term', 'Count'])

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('RdYlGn_r', len(top_words))
ax.barh(top_words['Term'][::-1], top_words['Count'][::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Most Frequent Terms in Paper Titles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'top_title_terms.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_13456\2494145457.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Abstract/Document Length Distribution

In [6]:
df['abstract_len'] = df['abstract'].fillna('').str.len()
df['doc_len'] = df['document'].fillna('').str.len()
df['word_count'] = df['document'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df['abstract_len'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(df['abstract_len'].median(), color='red', linestyle='--',
                label=f'Median: {df["abstract_len"].median():.0f}')
axes[0].set_xlabel('Abstract Length (characters)')
axes[0].set_ylabel('Number of Papers')
axes[0].set_title('Abstract Length Distribution', fontsize=13, fontweight='bold')
axes[0].legend()

axes[1].hist(df['word_count'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].axvline(df['word_count'].median(), color='blue', linestyle='--',
                label=f'Median: {df["word_count"].median():.0f}')
axes[1].set_xlabel('Document Word Count')
axes[1].set_ylabel('Number of Papers')
axes[1].set_title('Document Word Count Distribution', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'document_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_13456\3264092009.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Major mHealth Application Areas

In [7]:
# Identify broad application areas based on keywords in title + abstract
application_areas = {
    'Chronic Disease Management': ['diabetes', 'hypertension', 'cardiovascular', 'chronic', 'asthma', 'copd', 'heart failure'],
    'Mental Health': ['mental health', 'depression', 'anxiety', 'psychological', 'stress', 'psychiatric', 'wellbeing', 'well-being'],
    'Physical Activity & Fitness': ['physical activity', 'exercise', 'fitness', 'sedentary', 'step count', 'walking', 'activity tracker'],
    'Maternal & Child Health': ['maternal', 'pregnancy', 'prenatal', 'postnatal', 'neonatal', 'child health', 'pediatric', 'infant'],
    'Medication Adherence': ['medication adherence', 'drug adherence', 'treatment adherence', 'compliance', 'medication management'],
    'Remote Patient Monitoring': ['remote monitoring', 'remote patient', 'vital signs', 'home monitoring', 'telemonitoring'],
    'Wearable Sensors': ['wearable', 'sensor', 'accelerometer', 'gyroscope', 'smartwatch', 'biosensor', 'iot'],
    'Health Education': ['health education', 'health literacy', 'health promotion', 'health information', 'health awareness'],
    'Clinical Decision Support': ['clinical decision', 'decision support', 'diagnostic', 'screening', 'triage', 'clinical trial'],
    'Infectious Disease': ['infectious', 'covid', 'hiv', 'tuberculosis', 'malaria', 'pandemic', 'epidemic', 'vaccination'],
}

area_counts = {}
for area, keywords in application_areas.items():
    count = 0
    for _, row in df.iterrows():
        text = (str(row['title']) + ' ' + str(row['abstract'])).lower()
        if any(kw in text for kw in keywords):
            count += 1
    area_counts[area] = count

area_df = pd.DataFrame(list(area_counts.items()), columns=['Application Area', 'Count'])
area_df = area_df.sort_values('Count', ascending=False).reset_index(drop=True)

print('=== mHealth APPLICATION AREAS ===')
for _, row in area_df.iterrows():
    pct = 100 * row['Count'] / len(df)
    print(f'  {row["Application Area"]}: {row["Count"]} ({pct:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('Set2', len(area_df))
bars = ax.barh(area_df['Application Area'][::-1], area_df['Count'][::-1],
               color=colors[::-1], edgecolor='white')
ax.set_xlabel('Number of Papers')
ax.set_title('Major mHealth Application Areas in Dataset', fontsize=14, fontweight='bold')

for bar, val in zip(bars, area_df['Count'][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2.,
            str(val), ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'mhealth_application_areas.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

=== mHealth APPLICATION AREAS ===
  Chronic Disease Management: 475 (27.1%)
  Wearable Sensors: 450 (25.7%)
  Mental Health: 303 (17.3%)
  Clinical Decision Support: 292 (16.7%)
  Physical Activity & Fitness: 267 (15.3%)
  Remote Patient Monitoring: 246 (14.1%)
  Infectious Disease: 218 (12.5%)
  Health Education: 193 (11.0%)
  Medication Adherence: 138 (7.9%)
  Maternal & Child Health: 117 (6.7%)


Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_13456\225398859.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Journal Distribution

In [8]:
if 'journal' in df.columns:
    journals = df['journal'].fillna('').str.strip()
    journals = journals[journals != '']
    top_journals = journals.value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    colors = sns.color_palette('Spectral', len(top_journals))
    ax.barh(top_journals.index[::-1], top_journals.values[::-1],
            color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Number of Papers')
    ax.set_title('Top 15 Journals Publishing mHealth Research', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'top_journals.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved.')
else:
    print('Journal column not available.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_13456\2912977021.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Word Cloud of Titles

In [9]:
if HAS_WORDCLOUD:
    all_titles = ' '.join(df['title'].dropna().tolist())
    
    wc = WordCloud(
        width=1200, height=600,
        background_color='white',
        max_words=100,
        colormap='viridis',
        stopwords=stopwords,
        random_state=RANDOM_SEED
    ).generate(all_titles.lower())
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud of mHealth Paper Titles', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'title_wordcloud.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved.')
else:
    print('WordCloud library not installed. Skipping.')

WordCloud library not installed. Skipping.


## 8. Dataset Balance & Characteristics Summary

In [10]:
# Dataset characteristics summary
print('=== DATASET CHARACTERISTICS SUMMARY ===')
print(f'\nTotal relevant papers: {len(df)}')

# Year balance
print(f'\nYear range: {sorted([y for y in df["year"].unique() if str(y).isdigit()])}')
year_std = df[df['year'].str.match(r'^\d{4}$', na=False)]['year'].value_counts().std()
print(f'Year distribution std: {year_std:.1f} (lower = more balanced)')

# MeSH coverage
mesh_pct = (df['mesh_terms'].fillna('').str.strip() != '').mean() * 100
print(f'\nMeSH term coverage: {mesh_pct:.1f}%')

# Relevance category balance
print(f'\nRelevance category distribution:')
if 'relevance_category' in df.columns:
    rel_dist = df['relevance_category'].value_counts()
    for cat, count in rel_dist.items():
        print(f'  {cat}: {count} ({100*count/len(df):.1f}%)')

# Source balance
if 'source' in df.columns:
    print(f'\nData source distribution:')
    src_dist = df['source'].value_counts()
    for src, count in src_dist.items():
        print(f'  {src}: {count} ({100*count/len(df):.1f}%)')

# Potential imbalances
print(f'\n=== POTENTIAL IMBALANCES ===')
print(f'Most represented year: {df["year"].mode().values[0]} ({df["year"].value_counts().max()} papers)')
print(f'Least represented year: {df["year"].value_counts().idxmin()} ({df["year"].value_counts().min()} papers)')
print(f'Note: Recent years (2024-2026) may have incomplete data.')
print(f'Note: 2026 papers represent partial year data.')

print(f'\nEDA complete. All figures saved to: {FIGURES_DIR}')

=== DATASET CHARACTERISTICS SUMMARY ===

Total relevant papers: 1750

Year range: []
Year distribution std: nan (lower = more balanced)

MeSH term coverage: 67.4%

Relevance category distribution:
  Possibly relevant: 923 (52.7%)
  Relevant: 593 (33.9%)
  Highly relevant: 234 (13.4%)

Data source distribution:
  expanded: 1472 (84.1%)
  original: 278 (15.9%)

=== POTENTIAL IMBALANCES ===
Most represented year: 2024.0 (267 papers)
Least represented year: 2012.0 (6 papers)
Note: Recent years (2024-2026) may have incomplete data.
Note: 2026 papers represent partial year data.

EDA complete. All figures saved to: E:\MAJOR PROJECT\reports\figures


In [11]:
# Clean up
df = df.drop(columns=['abstract_len', 'doc_len', 'word_count'], errors='ignore')
print('Exploratory data analysis complete.')

Exploratory data analysis complete.
